# Dev Container Setup & Dependency Installation

This notebook verifies that the Dev Container environment is correctly configured and installs/validates all Python dependencies.

**Prerequisites:** Open this repository in VS Code and reopen in the Dev Container before running these cells.

---

## How to Use
Run each cell top-to-bottom. Cells with `%%bash` are shell commands. If a check fails, follow the fix noted in the output comment.

## 1. Verify Dev Container Environment

Confirm you are running inside the Dev Container and that the expected tools are present.

In [ ]:
%%bash
echo "=== OS ==="
cat /etc/os-release | grep PRETTY_NAME

echo ""
echo "=== Python ==="
python3 --version

echo ""
echo "=== pip ==="
pip --version

echo ""
echo "=== Working directory ==="
pwd

echo ""
echo "=== Workspace structure ==="
ls -1 /workspaces/marketing-model-mlops-azure/

In [ ]:
%%bash
echo "=== Azure CLI ==="
az --version 2>&1 | head -1

echo ""
echo "=== Docker CLI ==="
docker --version 2>&1 || echo "WARN: Docker CLI not found — check Dev Container feature config"

echo ""
echo "=== Git ==="
git --version

## 2. Install / Upgrade Dependencies

Dependencies are installed automatically on Dev Container start via `postCreateCommand`. Run this cell if you have updated `requirements.txt` or if packages are missing.

In [ ]:
%%bash
cd /workspaces/marketing-model-mlops-azure

echo "Installing dependencies from requirements.txt..."
pip install -r requirements.txt --quiet
echo "Done."

## 3. Verify Key Packages

Confirm the packages required for the ML pipeline and API are importable.

In [ ]:
packages = {
    "pandas": "Data manipulation",
    "numpy": "Numerical computing",
    "sklearn": "ML pipeline (scikit-learn)",
    "fastapi": "API framework",
    "uvicorn": "ASGI server",
    "pydantic": "Request validation",
    "joblib": "Model serialisation",
    "yaml": "Config parsing (pyyaml)",
    "pytest": "Test framework",
}

all_ok = True
for pkg, purpose in packages.items():
    try:
        __import__(pkg)
        print(f"  OK  {pkg:<12} — {purpose}")
    except ImportError:
        print(f"  MISSING  {pkg:<12} — {purpose}")
        all_ok = False

print()
if all_ok:
    print("All packages verified. Environment is ready.")
else:
    print("Some packages are missing. Re-run the pip install cell above.")

## 4. Verify Project Structure

Confirm all expected source files and directories are present.

In [ ]:
import os

root = "/workspaces/marketing-model-mlops-azure"

required_paths = [
    "main.py",
    "config.yaml",
    "requirements.txt",
    "Dockerfile",
    "src/data.py",
    "src/features.py",
    "src/train.py",
    "src/evaluate.py",
    "src/api/app.py",
    "data/raw/bank_marketing_data.csv",
    "tests/conftest.py",
    "tests/test_api.py",
    "tests/test_config.py",
    "tests/test_data.py",
    "tests/test_features.py",
    "k8s/deployment.yaml",
    "k8s/service.yaml",
    "k8s/deployment-dev.yaml",
    "k8s/service-dev.yaml",
    "k8s/quota-dev.yaml",
]

all_ok = True
for path in required_paths:
    full = os.path.join(root, path)
    exists = os.path.exists(full)
    status = "  OK " if exists else "  MISSING"
    if not exists:
        all_ok = False
    print(f"{status}  {path}")

print()
print("Project structure OK" if all_ok else "Some files are missing — check the repository.")

## 5. Verify Config Loads

Confirm `config.yaml` parses correctly and contains the expected keys.

In [ ]:
import sys
sys.path.insert(0, "/workspaces/marketing-model-mlops-azure")

from src.config import load_config

cfg = load_config("/workspaces/marketing-model-mlops-azure/config.yaml")

print("Config loaded successfully:")
import yaml
print(yaml.dump(cfg, default_flow_style=False))

---

## Summary

| Check | Expected result |
|---|---|
| OS | Debian GNU/Linux 12 (bookworm) |
| Python | 3.12.x |
| Docker CLI | Connects to host daemon |
| All packages | imported without error |
| Project structure | All paths present |
| Config | Loads and parses cleanly |

Once all cells pass, open **`03_ml_pipeline.ipynb`** to train the model and run tests.